In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))


## Cell 1 — clone + install (idempotent) + switch to Rainbow v2 branch

Pulls the latest code from GitHub (Rainbow v2: Dueling + Noisy Nets + Prioritized Replay), installs dependencies, and registers the Atari ROMs. Safe to re-run.

**Changes in this run vs vanilla DQN:**
- Dueling DQN: V(s) + A(s,a) head
- Noisy Nets: learned weight noise (no epsilon-greedy)
- Prioritized Replay: smart sampling by TD error
- 5M steps, 100k buffer, 100k/100k save/eval freq


In [2]:
import os, shutil, subprocess, sys

REPO = 'spaceinvaderrl'
REPO_DIR = f'/kaggle/working/{REPO}'
CKPT_DIR = f'{REPO_DIR}/runs'

# IMPORTANT: re-cloning the repo would nuke our checkpoint directory
# (runs/) and force the next run to start from scratch. Stash any existing
# runs/ to /kaggle/working/_runs_stash BEFORE the clone, restore AFTER.
_STASH = '/kaggle/working/_runs_stash'
if os.path.isdir(_STASH):
 shutil.rmtree(_STASH)
if os.path.isdir(f'{REPO_DIR}/runs'):
 shutil.move(f'{REPO_DIR}/runs', _STASH)
if os.path.isdir(REPO_DIR):
 shutil.rmtree(REPO_DIR)

os.chdir('/kaggle/working')
!git clone https://github.com/FunctionalFalcon/{REPO}.git

os.chdir(REPO_DIR)
!git pull origin main || echo 'pull skipped (offline?)'
# Switch to Rainbow v2 branch (Dueling + Noisy + Prioritized Replay)
!git fetch origin
!git checkout clean-main-v2
!git pull origin clean-main-v2

# Restore stashed runs/ (if any). The fresh repo has no runs/ directory,
# so this just merges the saved checkpoints back in.
if os.path.isdir(_STASH):
 if os.path.isdir(f'{REPO_DIR}/runs'):
  # shouldn't happen (repo has no runs/), but be defensive
  shutil.rmtree(f'{REPO_DIR}/runs')
  shutil.move(_STASH, f'{REPO_DIR}/runs')
  print(f'[resume] restored checkpoint directory from {_STASH}')
 else:
  shutil.move(_STASH, f'{REPO_DIR}/runs')
  print(f'[resume] restored checkpoint directory from {_STASH}')
else:
 os.makedirs(f'{REPO_DIR}/runs', exist_ok=True)

# === Resume from input dataset (if present) ===
# Kaggle auto-unzips uploaded datasets into /kaggle/input/.../spaceinvaderrl/runs/
# The trainer can't read from /kaggle/input (read-only) so we copy files
# into the working dir's runs/. Skip if a final.pt already exists in
# working-dir runs/ (means the input dataset has already been merged, OR
# a fresh run polluted the dir -- in either case, don't trample).
INPUT_RUNS = '/kaggle/input/datasets/ngocanhduong1111/spaceinvader-scratch-resume-5m/spaceinvaderrl/runs'
if os.path.isdir(INPUT_RUNS):
 if os.path.exists(f'{CKPT_DIR}/dqn_scratch_final.pt'):
  print(f'[resume] {CKPT_DIR}/dqn_scratch_final.pt already exists, NOT overwriting from input dataset')
 else:
  n, total_mb = 0, 0.0
  for fname in os.listdir(INPUT_RUNS):
   src = os.path.join(INPUT_RUNS, fname)
   dst = os.path.join(CKPT_DIR, fname)
   if os.path.isfile(src):
    shutil.copy2(src, dst)
    total_mb += os.path.getsize(dst) / 1e6
    n += 1
  print(f'[resume] copied {n} files ({total_mb:.1f} MB) from input dataset into {CKPT_DIR}')

# requirements.txt pins the modern stack (gymnasium 1.3.0 / shimmy 2.0.1 /
# ale-py 0.10.2). Don't override here - older pins (gymnasium 0.29.1, ale-py
# 0.8.x) aren't on PyPI for Python 3.12 and will hard-fail the install.
!pip install --no-cache-dir -q -r requirements.txt 2>&1 | tail -5
!pip show ale-py shimmy gymnasium 2>&1 | grep -E "^(Name|Version)"

import gymnasium as gym
import ale_py                          # ← add this
gym.register_envs(ale_py)             # ← and this
env = gym.make('ALE/SpaceInvaders-v5', frameskip=1, repeat_action_probability=0.25)
env.close()
print('envs registered OK')

Cloning into 'spaceinvaderrl'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 260 (delta 119), reused 147 (delta 61), pack-reused 54 (from 1)
Receiving objects: 100% (260/260), 216.73 MiB | 44.43 MiB/s, done.
Resolving deltas: 100% (130/130), done.
Updating files: 100% (50/50), done.
From https://github.com/FunctionalFalcon/spaceinvaderrl
 * branch            main       -> FETCH_HEAD
Already up to date.
Branch 'clean-main-v2' set up to track remote branch 'clean-main-v2' from 'origin'.
Switched to a new branch 'clean-main-v2'
From https://github.com/FunctionalFalcon/spaceinvaderrl
 * branch            clean-main-v2 -> FETCH_HEAD
Already up to date.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.
tensorflow 2.19.0 requir

A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]


## Cell 2 — smoketest (~3 min on T4)

Runs the from-scratch DQN for 5,000 steps to validate the whole pipeline before committing to the multi-hour run. Expected: a `.pt` saved under `runs/`, the eval CSV has 1 row, loss small and bounded (grad clipping caps spikes so anything < 5 is fine), no NaNs.

Pipeline: env_fixed with min_repeat=3 → Double DQN → NatureCNN → Huber loss + grad clipping → Adam.

In [3]:
os.chdir(REPO_DIR)
!python -m scratch.train --total_steps 5000 --save_freq 5000 --eval_freq 5000

import os
assert os.path.exists(f'{CKPT_DIR}/dqn_scratch_step_5000.pt'), 'no checkpoint'
assert os.path.exists(f'{CKPT_DIR}/dqn_scratch_eval.csv'), 'no eval csv'
print('\n[smoketest] OK')



[session] start_time = 2026-07-01 08:18:43
[session] pid = 158
[session] resume = (none, fresh run)
[setup] Using QNetwork (Dueling + Noisy Nets)
[setup] device = cuda (requested=auto, cuda_available=True)
[setup] GPU: Tesla T4
[setup] total_steps = 5,000
[setup] save_freq = 5,000, eval_freq = 5,000
[setup] train_freq = 4 (hardcoded to match paper)
[setup] min_repeat = 3 (MinActionRepeat wrapper applied via env_fixed)
[setup] auto_zip_freq = 200,000 (rescue zip every N steps, 0=off)
[setup] prio_alpha = 0.6, beta = 0.4→1.0 (IS correction ramps over first 50% of training)
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
[setup] Using PrioritizedReplayBuffer (alpha=0.6, beta=0.4→1.0)

[training] starting main loop...
 step   4,234/5,000 | noisy | loss     n/a | q[n/a, n/a, n/a] | ep    11 | R_last  110.0 | R_mean10  112.5 | buf  4,234 | 404.2 sps
 [ckpt] saved dqn_scratch_step_5000.pt
 [eval] step 5,000: mean_R =    91.5 +/- 21.7

[done] saved dqn_scratch_

## Cell 3 — full 8,000,000-step Rainbow v2 run on T4 (fresh start)

Wall time estimate on T4: ~60-90 min for 8M steps at 700-1000 sps.

Saves a .pt every 100k steps, runs 10 greedy eval episodes every 100k steps.
Auto-zips a rescue snapshot to /kaggle/working/runs.zip every 200k steps.

**Rainbow v2 architecture:**
- Dueling DQN: V(s) + A(s,a) head (Wang et al., 2016)
- Noisy Nets: learned weight noise replaces epsilon-greedy (Fortunato et al., 2017)
- Prioritized Replay: smart sampling by TD error via SumTree (Schaul et al., 2015)
- Double DQN: online picks action, target provides value

**Hyperparameters:**
- total_steps: 8,000,000
- buffer_size: 100,000
- prio_alpha: 0.6, prio_beta: 0.4 -> 1.0
- save_freq: 100,000, eval_freq: 100,000
- eval_episodes: 10


In [4]:
os.chdir(REPO_DIR)
# Fresh start with Rainbow v2 (Dueling + Noisy + Prioritized Replay)
# No resume flag = starts from scratch
!python -m scratch.train   --total_steps 8000000   --save_freq 100000   --eval_freq 100000



[session] start_time = 2026-07-01 08:19:21
[session] pid = 171
[session] resume = (none, fresh run)
[setup] Using QNetwork (Dueling + Noisy Nets)
[setup] device = cuda (requested=auto, cuda_available=True)
[setup] GPU: Tesla T4
[setup] total_steps = 8,000,000
[setup] save_freq = 100,000, eval_freq = 100,000
[setup] train_freq = 4 (hardcoded to match paper)
[setup] min_repeat = 3 (MinActionRepeat wrapper applied via env_fixed)
[setup] auto_zip_freq = 200,000 (rescue zip every N steps, 0=off)
[setup] prio_alpha = 0.6, beta = 0.4→1.0 (IS correction ramps over first 50% of training)
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
[setup] Using PrioritizedReplayBuffer (alpha=0.6, beta=0.4→1.0)

[training] starting main loop...
 step   4,335/8,000,000 | noisy | loss     n/a | q[n/a, n/a, n/a] | ep     6 | R_last  285.0 | R_mean 6  285.0 | buf  4,335 | 414.2 sps
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File

## Cell 4 — emergency resume (only if Cell 3 was interrupted)

If Kaggle disconnected you mid-run, the last completed checkpoint is what you resume from.
The --resume latest flag auto-picks the highest-step checkpoint in runs/.

Re-run Cell 4 until the total reaches 8,000,000.


In [ ]:
os.chdir(REPO_DIR)
# Auto-picks the most recent step checkpoint
!python -m scratch.train --resume latest --total_steps 8000000 --save_freq 100000 --eval_freq 100000


## Cell 5 — greedy evaluation (20 episodes)

Loads runs/dqn_scratch_final.pt and runs the Rainbow v2 agent (Dueling + Noisy + Prioritized Replay) greedily for 20 episodes. Reports mean +/- std episode reward.

Compare against:
- Vanilla DQN baseline: 257.5 +/- 104.9
- Previous from-scratch run: 697 +/- 232 at 8M steps
- Rainbow paper on Space Invaders: ~2000-3000 (full Rainbow, 50M steps)


In [16]:
os.chdir(REPO_DIR)
!python -m scratch.evaluate --checkpoint {CKPT_DIR}/dqn_scratch_final.pt --episodes 20


=== From-Scratch DQN Evaluation ===
 Checkpoint : /kaggle/working/spaceinvaderrl/runs/dqn_scratch_final.pt
 Episodes : 20
 Seed : 12345
 Device : cpu
 Loaded QNetwork (step=5,000).
 min_repeat = 3 (MinActionRepeat wrapper applied via env_fixed)

A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
Running 20 greedy episodes...
 ep   1/20: reward =     0.0
 ep   2/20: reward =     0.0
 ep   3/20: reward =     0.0
 ep   4/20: reward =     0.0
 ep   5/20: reward =     0.0
 ep   6/20: reward =     0.0
 ep   7/20: reward =     0.0
 ep   8/20: reward =     0.0
 ep   9/20: reward =     0.0
 ep  10/20: reward =     0.0
 ep  11/20: reward =     0.0
 ep  12/20: reward =     0.0
 ep  13/20: reward =     0.0
 ep  14/20: reward =     0.0
 ep  15/20: reward =     0.0
 ep  16/20: reward =     0.0
 ep  17/20: reward =     0.0
 ep  18/20: reward =     0.0
 ep  19/20: reward =     0.0
 ep  20/20: reward =     0.0

 Mean reward:      0.0 +/- 0.0

Compare against the SB3 baselin

## Cell 6 — record videos (one trained, one random baseline)

Records a 30fps MP4 of the Rainbow v2 trained agent (Dueling + Noisy DQN with Prioritized Replay) + a random-policy baseline.


In [17]:
%%writefile /tmp/record_scratch_trained.py
"""Record a video of the FROM-SCRATCH DQN agent playing Space Invaders.

Loads a torch .pt checkpoint (not SB3's .zip), rebuilds QNetwork, and
writes an MP4 via imageio (gymnasium's RecordVideo crashes on Kaggle
because of an fps=None bug).

Uses env_fixed with min_repeat pulled from the checkpoint's saved hp
dict so the recorded env matches the training env exactly.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

# Insert the repo root at the front of sys.path so the absolute imports
# below (`from scratch.X`, `from shared.X`) resolve regardless of CWD.
# Without this, running this script from /tmp fails with
# `ModuleNotFoundError: No module named 'scratch'`.
_REPO = Path('/kaggle/working/spaceinvaderrl')
if str(_REPO) not in sys.path:
 sys.path.insert(0, str(_REPO))

import imageio.v2 as imageio
import torch

from scratch.network import QNetwork
from scratch.hyperparam import Hyperparameters
from scratch.evaluate import greedy_action
from shared.preprocessing import env_fixed

VIDEO_DIR = _REPO / 'videos'

def main():
	p = argparse.ArgumentParser()
	p.add_argument('--checkpoint', type=str, required=True)
	p.add_argument('--episodes', type=int, default=1)
	p.add_argument('--name-prefix', type=str, default='trained-scratch')
	p.add_argument('--seed', type=int, default=42)
	p.add_argument('--fps', type=int, default=30)
	args = p.parse_args()

	VIDEO_DIR.mkdir(exist_ok=True)

	ckpt = torch.load(args.checkpoint, map_location='cpu', weights_only=False)
	hp = Hyperparameters(**ckpt.get('hp', {}))
	num_actions = int(getattr(hp, 'num_actions', 6))
	min_repeat = int(getattr(hp, 'min_repeat', 4))
	q_net = QNetwork(num_actions=num_actions)
	q_net.load_state_dict(ckpt['model_state_dict'])
	q_net.eval()

	env = env_fixed(seed=args.seed, render_mode='rgb_array', min_repeat=min_repeat)
	for ep in range(args.episodes):
		out_path = VIDEO_DIR / f'{args.name_prefix}-episode-{ep}.mp4'
		print(f' Recording episode {ep+1} -> {out_path}')
		obs, _ = env.reset()
		total, steps, done = 0.0, 0, False
		with imageio.get_writer(out_path, fps=args.fps) as writer:
			while not done:
				writer.append_data(env.render())
				action = greedy_action(q_net, obs, device='cpu')
				obs, r, term, trunc, _ = env.step(action)
				total += float(r)
				steps += 1
				done = term or trunc
		print(f' Episode {ep+1}: {steps} steps, reward={total:.1f}')
		print(f' saved: {out_path}')
	env.close()
	return 0

if __name__ == '__main__':
	sys.exit(main())

Overwriting /tmp/record_scratch_trained.py


In [ ]:
import os, subprocess, sys
os.chdir(REPO_DIR)

# Wipe the videos dir before recording so old files from previous sessions
# don't pollute the listing. shutil.rmtree is more reliable than a
# hand-rolled loop because it handles files + subdirs + read-only entries.
vdir = f'{REPO_DIR}/videos'
if os.path.isdir(vdir):
    shutil.rmtree(vdir)
os.makedirs(vdir, exist_ok=True)

# Both scripts do absolute imports of `from scratch.X` and
# `from shared.X`. Those resolve only when CWD is the repo root
# (parent of scratch/ and shared/). Passing cwd=REPO_DIR makes
# sys.path include REPO_DIR so the imports work.
res = subprocess.run([
    sys.executable, '/tmp/record_scratch_trained.py',
    '--checkpoint', f'{CKPT_DIR}/dqn_scratch_final.pt',
    '--episodes', '1',
], cwd=REPO_DIR)
if res.returncode != 0:
    print('[record_trained] FAILED - see traceback above.')

# Random baseline: keep using make_env (no MinActionRepeat) so the
# comparison video shows raw random behavior, not random-with-commit.
res = subprocess.run([
    sys.executable, f'{REPO_DIR}/sb3/record_random.py',
    '--episodes', '1',
], cwd=REPO_DIR)
if res.returncode != 0:
    print('[record_random] FAILED - see traceback above.')

print('\n[videos] written to:', vdir)
for n in sorted(os.listdir(vdir)):
    full = os.path.join(vdir, n)
    print(f'  {n} ({os.path.getsize(full):,} bytes)')

## Cell 7 â€” zip everything for download

Kaggle's UI lets you download anything in `/kaggle/working`. This zips the videos + runs + logs into a single file you can grab at the end.

In [19]:
import os
os.chdir('/kaggle/working')
out = '/kaggle/working/runs.zip'
if os.path.exists(out):
 os.remove(out)
# Pack everything you'd want to download into one file:
# - {REPO}/videos: .mp4 recordings from Cell 6 (may not exist if Cell 6
#   didn't run; zip -rq will skip with no error)
# - {REPO}/runs: step_*.pt, final.pt, eval.csv, metrics.csv, train.log
# We do NOT include the rescue runs.zip at the working dir root to avoid
# zip-into-self errors (it's a duplicate of the curated content above
# anyway, plus 5 most-recent step ckpts + CSVs).
!zip -rq {out} {REPO}/videos {REPO}/runs 2>&1 | tail -5 || true
print(f'\n[done] {out} ({os.path.getsize(out):,} bytes)')
print('Download this file via the Kaggle file browser to keep your trained model.')


[done] /kaggle/working/runs.zip (873,467,662 bytes)
Download this file via the Kaggle file browser to keep your trained model.
